# ③片麻痺閾値論文(10524680)— 査読対応のための追加検証ノートブック

元のノートブック `g1_hemiparesis_threshold.ipynb` と**同じ学習済みcheckpoint(`logs/rsl_rl/g1_velocity/*_seed{1..8}`)**を使います。
元の結果CSV(`g1_hemiparesis_fine.csv` など)が同じフォルダにあれば、整合性チェックにも使います(無くても動きます)。

| 節 | 内容 | 対応する査読コメント |
|---|---|---|
| §A | 行動生成パイプラインの確認(関節順序・スケール・クリップ)と、「最終層の重み・バイアスをα倍」=「行動をちょうどα倍」の数値的な確認 | R1-1(減衰の数式), R2(G1仕様) |
| §B | **旧§9の「デフォルト姿勢に0.0°で一致」が、自動リセット後の状態を読んでいたアーティファクトであることの実証** | R1-4, R2(collapseは組み込まれた挙動では?) |
| §C | 正しい計測(リセット前の状態)による機序の再解析(8seed×2条件×6水準) | R1-4, R2 |
| §D | 初期条件を変えた反復試行による、policy間 vs policy内のばらつきの分離 | R1-2 |
| §A2 | **(追加)** 仕様の記録:総質量・ソフトウェアのバージョン・腰の3関節・デフォルト姿勢・PDゲイン(MuJoCoモデルから直接読み出し) | R2-4, R1-4, R2-2 |
| §D2 | **(追加・任意)** 反復試行を他の3速度(0.3・0.7・0.9 m/s)にも拡張 | R1-2, R2-1(速度をまたぐ非対称性の頑健性) |
| §E | 統計(符号検定・Wilcoxon正確検定・ブートストラップCI・分散分解) | R1-2 |
| §E2 | **(追加)** 速度ごとの左右差の統計(元の速度依存CSV、§D2の結果) | R1-2(抄録の「全速度で非対称」の訂正) |
| §F | **(追加)** 改訂原稿用の図(用量反応曲線+95% CI、転倒前の関節角の時間経過) | R1-2, R1-4 |
| §G | **(追加)** GitHub公開用:各seedの最終checkpointの確認とコピー | R1-5 |
| §H | **(追加)** 粗いスイープと速度依存スイープを改訂時の環境で測り直し(Table 1・3と図2の更新) | 表どうしの整合性 |

**実行の目安(T4)**:§A〜§C・§E・§A2・§F・§Gは短時間です。§Dは80ロールアウト、§D2は240ロールアウト、§Hは416試行で時間がかかります
(どちらも途中から再開できます)。まず§D2以外を実行し、時間に余裕があれば§D2を実行してください。

**重要な設計上の注意**: §C・§Dでは、環境の`fell_over`終了条件を**外し**、同じ判定式(体幹の傾き>70°)を
ノートブック側で毎ステップ計算して転倒を検出します。これにより、転倒時に環境が自動リセットされることがなく、
**転倒した瞬間の、本当の関節角**を記録できます(終了条件は環境のリセットにしか使われないため、
転倒までの力学は元の実験と完全に同一です)。

## 1. セットアップ

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = ""
os.environ["WANDB_MODE"] = "offline"

%pip install -q "mjlab==1.2.0" "mujoco==3.5.0" "warp-lang==1.12.1" "numpy>=2.0,<2.3" scipy mediapy pandas

import torch
assert torch.cuda.is_available(), "GPU ランタイムに切り替えてください"
print("✅ install done:", torch.cuda.get_device_name(0))

### 1-2. 学習済みcheckpointと元の結果CSVを、Googleドライブから復元する

新しいランタイムには `logs/` フォルダ(学習済みcheckpoint)がありません。学習ノートブックが退避した
`MyDrive/g1_cross_seed_logs/logs` を、作業フォルダ(`/content`)にコピーします。
あわせて、元の実験の結果CSV(`g1_hemiparesis_fine.csv` など)を、ドライブ内のよく使う場所(マイドライブ直下・g1_cross_seed_logs・Colab Notebooks など)から探して、同じ場所にコピーします。

**実行すると、Googleアカウントへのアクセス許可を求める画面が出ます。学習に使ったのと同じアカウントを選んで「許可」してください。**

In [ ]:
import os, glob, shutil
from google.colab import drive
drive.mount("/content/drive")
os.chdir("/content")

DRIVE_LOGS = "/content/drive/MyDrive/g1_cross_seed_logs/logs"   # 学習ノートブックの退避先
if not os.path.isdir("logs/rsl_rl/g1_velocity"):
    if os.path.isdir(DRIVE_LOGS):
        print("ドライブから logs をコピーしています(数分かかることがあります)...")
        shutil.copytree(DRIVE_LOGS, "/content/logs", dirs_exist_ok=True)
    else:
        print(f"⚠️ {DRIVE_LOGS} が見つかりません。ドライブ内の場所を確認し、DRIVE_LOGS を書き換えてください。")

found = sorted(glob.glob("logs/rsl_rl/g1_velocity/*_seed*"))
print(f"見つかった実行フォルダ: {len(found)} 個")
for s in range(1, 9):
    print(f"  seed{s}: {'OK' if any(p.endswith(f'_seed{s}') for p in found) else '見つかりません'}")

# 元の結果CSVを、ドライブ内のよく使う場所から探してコピーする(見つからなくても、この後の解析は動きます)
for name in ["g1_hemiparesis_fine.csv", "g1_hemiparesis_speed_dependence.csv", "g1_hemiparesis_coarse.csv"]:
    if os.path.exists(name):
        print(f"{name}: 既にあります"); continue
    # ドライブ全体を再帰的に探すと非常に遅いため、よく使う場所だけを探す
    cands = ["/content/drive/MyDrive", "/content/drive/MyDrive/g1_cross_seed_logs",
             "/content/drive/MyDrive/g1_hemiparesis", "/content/drive/MyDrive/Colab Notebooks"]
    hits = [os.path.join(c, name) for c in cands if os.path.exists(os.path.join(c, name))]
    if hits:
        shutil.copy(hits[0], name); print(f"{name}: {hits[0]} からコピーしました")
    else:
        print(f"{name}: ドライブ内に見つかりません(手動でアップロードしてください)")

In [ ]:
import math, glob, re, json, time
import numpy as np
import pandas as pd
from pathlib import Path
from dataclasses import asdict
from mjlab.envs import ManagerBasedRlEnv
from mjlab.rl import MjlabOnPolicyRunner, RslRlVecEnvWrapper
from mjlab.tasks.registry import load_env_cfg, load_rl_cfg

TASK = "Mjlab-Velocity-Flat-Unitree-G1"
DEVICE = "cuda:0"
ENV_SEED = 100
RIGHT_FACING_YAW_RAD = -math.pi / 2
FALL_LIMIT_RAD = math.radians(70.0)   # mjlab velocity task の fell_over と同じ閾値
SEEDS = [1, 2, 3, 4, 5, 6, 7, 8]
STEPS = 250
OUT_DIR = Path("revision_outputs"); OUT_DIR.mkdir(exist_ok=True)

def latest_ckpt(seed):
    run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
    assert run_dirs, f"seed{seed}: runディレクトリが見つかりません"
    return max(Path(run_dirs[-1]).glob("model_*.pt"),
               key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))

_probe_cfg = load_env_cfg(TASK, play=True)
_probe_cfg.scene.num_envs = 1
_probe_env = ManagerBasedRlEnv(cfg=_probe_cfg, device=DEVICE, render_mode=None)
JOINT_NAMES = list(_probe_env.scene["robot"].joint_names)
LEFT_HALF_JOINTS = [n for n in JOINT_NAMES if n.startswith("left_")]
RIGHT_HALF_JOINTS = [n for n in JOINT_NAMES if n.startswith("right_")]
assert len(JOINT_NAMES) == 29 and len(LEFT_HALF_JOINTS) == 13 and len(RIGHT_HALF_JOINTS) == 13
HEMI_CONDITIONS = {
    "right_half_paresis": [JOINT_NAMES.index(n) for n in RIGHT_HALF_JOINTS],
    "left_half_paresis":  [JOINT_NAMES.index(n) for n in LEFT_HALF_JOINTS],
}
_probe_env.close(); del _probe_env; torch.cuda.empty_cache()
print("中央(体幹)の関節:", [n for n in JOINT_NAMES if not n.startswith(("left_", "right_"))])

In [ ]:
def make_env_and_policy(checkpoint_path, joint_idx_list, level, env_seed=ENV_SEED, num_envs=1,
                        remove_fall_termination=False, obs_noise=False):
    # 元のノートブックと同じ構成。remove_fall_termination=True のときだけ fell_over 終了条件を外す。
    play_cfg = load_env_cfg(TASK, play=True)
    play_cfg.scene.num_envs = num_envs
    play_cfg.seed = env_seed
    play_cfg.events["reset_base"].params["pose_range"]["yaw"] = (RIGHT_FACING_YAW_RAD, RIGHT_FACING_YAW_RAD)
    if remove_fall_termination:
        play_cfg.terminations.pop("fell_over", None)
    if obs_noise:
        play_cfg.observations["actor"].enable_corruption = True
    agent_cfg = load_rl_cfg(TASK)

    raw_env = ManagerBasedRlEnv(cfg=play_cfg, device=DEVICE, render_mode=None)
    env = RslRlVecEnvWrapper(raw_env, clip_actions=agent_cfg.clip_actions)
    runner = MjlabOnPolicyRunner(env, asdict(agent_cfg), device=DEVICE)
    runner.load(str(checkpoint_path), load_cfg={"actor": True}, strict=True, map_location=DEVICE)
    policy = runner.get_inference_policy(device=DEVICE)
    with torch.no_grad():
        final_layer = runner.alg.actor.mlp[-1]
        for idx in joint_idx_list:
            final_layer.weight[idx, :] *= level
            final_layer.bias[idx] *= level
    return runner, policy, env, raw_env, agent_cfg

def close_all(*objs):
    for o in objs:
        try:
            o.close()
        except Exception:
            pass
    torch.cuda.empty_cache()

def tilt_rad(raw_env):
    pg = raw_env.scene["robot"].data.projected_gravity_b
    return torch.acos((-pg[:, 2]).clamp(-1.0, 1.0)).abs()

def joint_targets(raw_env):
    term = raw_env.action_manager.get_term("joint_pos")
    return term.raw_action * term.scale + term.offset

## §A. 行動生成パイプラインの確認(R1-1, R2)

論文のMethodsに書く「行動生成の式」を、実際の環境・checkpointから確認します。

- actorの最終層は**活性化関数なしの線形層**(`Linear(128→29)`)で、行動のクリップは無し(`clip_actions=None`)。
- 関節位置指令は `q_target = q_default + s_j × a_j`(`JointPositionAction`, `use_default_offset=True`)。
- よって、重み行・バイアスをα倍すると **a_j がちょうどα倍** → **指令の「デフォルト姿勢からのずれ」がα倍**。
  トルクの上限(筋力)を下げる操作ではない、という点を論文で明記します。

In [ ]:
ckpt1 = latest_ckpt(1)
runner, policy, env, raw_env, agent_cfg = make_env_and_policy(ckpt1, [], 1.0)
term = raw_env.action_manager.get_term("joint_pos")
robot = raw_env.scene["robot"]

print("最終層:", runner.alg.actor.mlp[-1], "/ 直前の層:", runner.alg.actor.mlp[-2])
print("clip_actions:", agent_cfg.clip_actions)
print("制御周期: dt = %.4f s (physics %.4f s × decimation %d) → %d Hz"
      % (raw_env.step_dt, raw_env.physics_dt, raw_env.cfg.decimation, round(1 / raw_env.step_dt)))
print("250ステップ =", 250 * raw_env.step_dt, "s")
print("行動の関節順序 == JOINT_NAMES :", list(term.target_names) == JOINT_NAMES)

scale = term.scale if torch.is_tensor(term.scale) else torch.full((29,), float(term.scale))
scale = scale.reshape(-1, 29)[0].detach().cpu().numpy()
q0 = robot.data.default_joint_pos[0].detach().cpu().numpy()
pipe_df = pd.DataFrame({"joint": JOINT_NAMES, "action_scale_rad": scale, "default_pos_deg": np.degrees(q0)})
print(pipe_df.to_string(index=False))
pipe_df.to_csv(OUT_DIR / "A_action_pipeline.csv", index=False)

# 数値的な等価性チェック: 同じ観測に対し、(1) 元のpolicy出力×α と (2) 最終層をα倍したpolicyの出力 を比較
obs = env.get_observations()
with torch.no_grad():
    a_orig = policy(obs).clone()
    alpha = 0.65
    idx = HEMI_CONDITIONS["right_half_paresis"]
    fl = runner.alg.actor.mlp[-1]
    fl.weight[idx, :] *= alpha; fl.bias[idx] *= alpha
    a_scaled = policy(obs).clone()
mask = torch.zeros(29, dtype=torch.bool, device=a_orig.device); mask[idx] = True
err_target = (a_scaled[:, mask] - alpha * a_orig[:, mask]).abs().max().item()
err_other = (a_scaled[:, ~mask] - a_orig[:, ~mask]).abs().max().item()
print(f"\n対象13関節: max|a_scaled - α·a_orig| = {err_target:.2e}")
print(f"非対象16関節: max|a_scaled - a_orig| = {err_other:.2e}")
close_all(env)
del runner, policy, env, raw_env

## §A2. 仕様の記録(R2-4, R1-4, R2-2)

回答レターの仕様表(R2-4)とSupplementary S3/S4に載せる値を、実行中の環境・MuJoCoモデルから直接読み出して記録します。
計算で求めていたPDゲイン・トルク上限・行動スケールが、実際のモデルの値と一致するかもここで確認します。

In [ ]:
import importlib.metadata as md_
import mujoco

ckpt1 = latest_ckpt(1)
runner, policy, env, raw_env, agent_cfg = make_env_and_policy(ckpt1, [], 1.0)
m = raw_env.sim.mj_model
robot = raw_env.scene["robot"]
term = raw_env.action_manager.get_term("joint_pos")

spec = {}
for pkg in ["mjlab", "mujoco", "mujoco-warp", "warp-lang", "rsl-rl-lib", "torch", "numpy"]:
    try:
        spec[f"version_{pkg}"] = md_.version(pkg)
    except Exception:
        spec[f"version_{pkg}"] = "not installed"
spec["torch.__version__"] = torch.__version__
spec["gpu"] = torch.cuda.get_device_name(0)
spec["total_mass_kg"] = float(mujoco.mj_getTotalmass(m))
spec["physics_dt_s"] = float(raw_env.physics_dt)
spec["control_dt_s"] = float(raw_env.step_dt)
spec["decimation"] = int(raw_env.cfg.decimation)
spec["n_joints"] = len(JOINT_NAMES)
spec["unattenuated_center_joints"] = [n for n in JOINT_NAMES if not n.startswith(("left_", "right_"))]
spec["actor_obs_dim"] = int(runner.alg.actor.mlp[0].in_features)
spec["actor_layers"] = str(runner.alg.actor.mlp)
spec["clip_actions"] = agent_cfg.clip_actions
spec["checkpoint_seed1"] = str(ckpt1)
for k, v in spec.items():
    print(f"{k:28s}: {v}")
with open(OUT_DIR / "A2_spec.json", "w") as f:
    json.dump(spec, f, indent=2, ensure_ascii=False, default=str)

# 関節ごと:デフォルト角・PDゲイン・トルク上限・行動スケール(MuJoCoモデルの値)
rows = []
scale = term.scale if torch.is_tensor(term.scale) else torch.full((29,), float(term.scale))
scale = scale.reshape(-1, 29)[0].detach().cpu().numpy()
q0 = robot.data.default_joint_pos[0].detach().cpu().numpy()
for j, name in enumerate(JOINT_NAMES):
    # 関節名に対応するactuatorを探す(mjlabのactuator名は関節名を含む)
    aid = [a for a in range(m.nu) if name in (mujoco.mj_id2name(m, mujoco.mjtObj.mjOBJ_ACTUATOR, a) or "")]
    if aid:
        a = aid[0]
        kp = float(m.actuator_gainprm[a, 0]); kd = float(-m.actuator_biasprm[a, 2])
        frc = m.actuator_forcerange[a].tolist() if m.actuator_forcelimited[a] else None
    else:
        kp = kd = np.nan; frc = None
    rows.append({"joint": name, "default_rad": q0[j], "default_deg": np.degrees(q0[j]),
                 "Kp_Nm_per_rad": kp, "Kd_Nms_per_rad": kd, "force_range_Nm": frc,
                 "action_scale_rad": scale[j]})
spec_df = pd.DataFrame(rows)
print(spec_df.round(3).to_string(index=False))
spec_df.to_csv(OUT_DIR / "A2_joint_table.csv", index=False)
close_all(env)
del runner, policy, env, raw_env

## §B. 旧§9のアーティファクトの実証(R1-4, R2)

旧§9のコードは `env.step()` の**後**に関節角を読んでいました。mjlab 1.2.0 の `ManagerBasedRlEnv.step()` は、
終了条件が成立した環境を**step()の内部でリセット**し(`reset_joints_by_offset`, `position_range=(0,0)` → 厳密にデフォルト姿勢)、
その後で戻ります。したがって、転倒ステップで読んだ値は「リセット後のデフォルト姿勢」そのものです。

同じ試行(seed1、右半身65%)を、(1) 旧方式 と (2) 終了条件を外してリセット前の状態を読む方式、で比較します。

In [ ]:
def rollout_old_style(ckpt, idx_list, level, speed=0.5):
    runner, policy, env, raw_env, _ = make_env_and_policy(ckpt, idx_list, level)
    cmd = torch.tensor([[speed, 0.0, 0.0]], device=DEVICE)
    obs = env.get_observations(); q_last = None; fell = False
    for step in range(STEPS):
        raw_env.command_manager.get_term("twist").vel_command_b[:] = cmd
        with torch.no_grad():
            obs, _, dones, _ = env.step(policy(obs))
        q_last = raw_env.scene["robot"].data.joint_pos[0].detach().cpu().numpy().copy()
        if bool(dones[0].item()):
            fell = True; break
    q0 = raw_env.scene["robot"].data.default_joint_pos[0].detach().cpu().numpy()
    close_all(env); del runner, policy, env, raw_env
    return fell, step + 1, np.degrees(np.abs(q_last - q0))

def rollout_logged(ckpt, idx_list, level, speed=0.5, env_seed=ENV_SEED, num_envs=1, obs_noise=False, keep_traj=True):
    # fell_over終了条件を外し、同じ判定式で転倒を検出。リセットは起きないので、転倒時の本当の状態が読める。
    runner, policy, env, raw_env, _ = make_env_and_policy(ckpt, idx_list, level, env_seed=env_seed,
                                                          num_envs=num_envs, remove_fall_termination=True,
                                                          obs_noise=obs_noise)
    robot = raw_env.scene["robot"]
    cmd = torch.tensor([[speed, 0.0, 0.0]], device=DEVICE)
    q0 = robot.data.default_joint_pos.detach().cpu().numpy().copy()          # (N,29)
    obs = env.get_observations()
    fall_step = np.full(num_envs, -1, dtype=int)
    Q, T, TILT, H = [], [], [], []
    for step in range(STEPS):
        raw_env.command_manager.get_term("twist").vel_command_b[:] = cmd
        with torch.no_grad():
            obs, _, _, _ = env.step(policy(obs))
        tilt = tilt_rad(raw_env).detach().cpu().numpy()
        newly = (fall_step < 0) & (tilt > FALL_LIMIT_RAD)
        fall_step[newly] = step + 1                      # 元の n_steps_completed と同じ数え方
        if keep_traj:
            Q.append(robot.data.joint_pos.detach().cpu().numpy().copy())
            T.append(joint_targets(raw_env).detach().cpu().numpy().copy())
            TILT.append(tilt.copy())
            H.append(robot.data.root_link_pos_w[:, 2].detach().cpu().numpy().copy())
        if (fall_step > 0).all():
            break
    close_all(env); del runner, policy, env, raw_env
    traj = None
    if keep_traj:
        traj = {"q": np.stack(Q, 1), "target": np.stack(T, 1), "tilt": np.stack(TILT, 1),
                "height": np.stack(H, 1), "q0": q0}
    return fall_step, traj

ckpt = latest_ckpt(1); idx = HEMI_CONDITIONS["right_half_paresis"]
fell_old, n_old, absdiff_old = rollout_old_style(ckpt, idx, 0.65)
fs, tr = rollout_logged(ckpt, idx, 0.65)
k = fs[0] - 1
absdiff_new = np.degrees(np.abs(tr["q"][0, k] - tr["q0"][0]))
print(f"旧方式: 転倒={fell_old}, n_steps={n_old}, 終端のデフォルトとの差 平均 {absdiff_old.mean():.4f}°, 最大 {absdiff_old.max():.4f}°")
print(f"新方式: 転倒ステップ={fs[0]}, 転倒時のデフォルトとの差 平均 {absdiff_new.mean():.2f}°, 最大 {absdiff_new.max():.2f}°")
print(f"新方式: 転倒1ステップ前の差 平均 {np.degrees(np.abs(tr['q'][0, k-1] - tr['q0'][0])).mean():.2f}°")
print("\n→ 旧方式が 0.0° になり、新方式でそうならなければ、旧§9の知見はリセット由来のアーティファクトと確定。")
print("  (n_steps が一致すること=終了条件を外しても転倒までの力学が同一であること、も確認)")

## §C. 正しい計測による機序の再解析(R1-4, R2)

8seed × 2条件 × 6水準(1.00=ベースライン, 0.75, 0.70, 0.65, 0.60, 0.55)、0.5 m/s。
各試行について、麻痺側13関節・非麻痺側16関節それぞれで、

- **実際の関節角**のデフォルトからのずれ `|q − q0|`
- **関節指令**のデフォルトからのずれ `|q_target − q0|`(減衰の直接の効果は、麻痺側の指令がデフォルトへ縮むこと)

を、(i) 転倒時点、(ii) 転倒前0.5 s(25ステップ)の平均、(iii) 非転倒試行では定常歩行区間(50–250ステップ)の平均、で記録します。
あわせて、元のCSVがあれば、転倒の有無と転倒ステップが一致するかを確認します(±1ステップは派生量の更新タイミングの差)。

In [ ]:
MECH_LEVELS = [1.00, 0.75, 0.70, 0.65, 0.60, 0.55]
MECH_COLS = ["condition", "seed", "level", "fell_over", "fall_step",
             "q_paretic_at_fall", "q_nonparetic_at_fall", "tgt_paretic_at_fall", "tgt_nonparetic_at_fall",
             "q_paretic_pre05", "q_nonparetic_pre05", "tgt_paretic_pre05", "tgt_nonparetic_pre05",
             "q_paretic_walk", "q_nonparetic_walk", "tgt_paretic_walk", "tgt_nonparetic_walk"]
MECH_CSV = OUT_DIR / "C_mechanism_summary.csv"
TRAJ_DIR = OUT_DIR / "C_trajectories"; TRAJ_DIR.mkdir(exist_ok=True)

def repair_mech_csv(path):
    # 旧版ノートブックは、転倒あり(13列)と転倒なし(9列)で列数の違う行を同じCSVに書いていた。列数で見分けて正しい表に直す
    import csv
    FALL = MECH_COLS[:13]
    WALK = MECH_COLS[:5] + MECH_COLS[13:]
    with open(path) as f:
        lines = list(csv.reader(f))
    if len(lines) == 0 or lines[0] == MECH_COLS:
        return
    recs = []
    for r in lines[1:]:
        if len(r) == len(FALL):
            recs.append(dict(zip(FALL, r)))
        elif len(r) == len(WALK):
            recs.append(dict(zip(WALK, r)))
    df = pd.DataFrame(recs).reindex(columns=MECH_COLS).replace("", np.nan)
    df["fell_over"] = df["fell_over"].map({"True": True, "False": False})
    for c in MECH_COLS[1:]:
        if c != "fell_over":
            df[c] = pd.to_numeric(df[c])
    path.rename(path.with_suffix(".old.csv"))
    df.to_csv(path, index=False)
    print(f"{path} を修復しました({len(df)}行)。元のファイルは {path.with_suffix('.old.csv')} に残しています。")

done = set()
if MECH_CSV.exists():
    repair_mech_csv(MECH_CSV)
    _d = pd.read_csv(MECH_CSV); done = set(zip(_d["condition"], _d["seed"], _d["level"]))

def side_stats(arr_deg, paretic_idx):
    m = np.zeros(29, dtype=bool); m[paretic_idx] = True
    return arr_deg[..., m].mean(-1), arr_deg[..., ~m].mean(-1)

for cond, idx in HEMI_CONDITIONS.items():
    for seed in SEEDS:
        ckpt = latest_ckpt(seed)
        for level in MECH_LEVELS:
            if (cond, seed, level) in done:
                continue
            t0 = time.time()
            fs, tr = rollout_logged(ckpt, idx, level)
            np.savez_compressed(TRAJ_DIR / f"{cond}_seed{seed}_L{int(round(level*100))}.npz", **tr)
            dq = np.degrees(np.abs(tr["q"][0] - tr["q0"][0]))          # (T,29)
            dt_ = np.degrees(np.abs(tr["target"][0] - tr["q0"][0]))
            qp, qn = side_stats(dq, idx); tp, tn = side_stats(dt_, idx)
            fell = fs[0] > 0
            row = {"condition": cond, "seed": seed, "level": level, "fell_over": fell,
                   "fall_step": int(fs[0]) if fell else np.nan}
            if fell:
                k = fs[0] - 1; w = slice(max(0, k - 24), k + 1)
                row.update(q_paretic_at_fall=qp[k], q_nonparetic_at_fall=qn[k],
                           tgt_paretic_at_fall=tp[k], tgt_nonparetic_at_fall=tn[k],
                           q_paretic_pre05=qp[w].mean(), q_nonparetic_pre05=qn[w].mean(),
                           tgt_paretic_pre05=tp[w].mean(), tgt_nonparetic_pre05=tn[w].mean())
            else:
                w = slice(50, None)
                row.update(q_paretic_walk=qp[w].mean(), q_nonparetic_walk=qn[w].mean(),
                           tgt_paretic_walk=tp[w].mean(), tgt_nonparetic_walk=tn[w].mean())
            pd.DataFrame([row]).reindex(columns=MECH_COLS).to_csv(MECH_CSV, mode="a", header=not MECH_CSV.exists(), index=False)
            print(f"{cond} seed{seed} L={level}: fell={fell} step={row['fall_step']} ({time.time()-t0:.0f}s)")
print("完了")

In [ ]:
mech = pd.read_csv(MECH_CSV)
pd.set_option("display.width", 200); pd.set_option("display.precision", 2)

# 元のCSVとの整合性(あれば)
if Path("g1_hemiparesis_fine.csv").exists():
    fine_orig = pd.read_csv("g1_hemiparesis_fine.csv")
    m = mech.merge(fine_orig, on=["condition", "seed", "level"], suffixes=("_new", "_orig"))
    same_fall = (m["fell_over_new"] == m["fell_over_orig"]).mean()
    step_diff = (m.loc[m["fell_over_orig"], "fall_step"] - m.loc[m["fell_over_orig"], "n_steps_completed"]).abs()
    print(f"元のfine CSVとの一致: 転倒有無 {same_fall*100:.1f}% 一致, 転倒ステップの差 最大 {step_diff.max() if len(step_diff) else 0} ステップ")
else:
    print("g1_hemiparesis_fine.csv が無いため、整合性チェックはスキップ")

print("\n=== 転倒率(本計測, 8seed) ===")
print(mech.pivot_table(index="level", columns="condition", values="fell_over", aggfunc="mean").sort_index(ascending=False))

base = mech[mech["level"] == 1.0]
print("\n=== ベースライン(100%, 非転倒)定常歩行中の |q−q0| 平均(度) ===")
print(base.groupby("condition")[["q_paretic_walk", "q_nonparetic_walk", "tgt_paretic_walk", "tgt_nonparetic_walk"]].mean())

falls = mech[mech["fell_over"]]
print(f"\n=== 転倒試行(n={len(falls)}): 転倒時・転倒前0.5sの |q−q0| と |q_target−q0|(度) ===")
cols = ["q_paretic_at_fall", "q_nonparetic_at_fall", "tgt_paretic_at_fall", "tgt_nonparetic_at_fall",
        "q_paretic_pre05", "q_nonparetic_pre05", "tgt_paretic_pre05", "tgt_nonparetic_pre05"]
print(falls.groupby(["condition", "level"])[cols].agg(["mean", "std"]).round(2).to_string())
print(f"\n転倒時に |q−q0| < 0.5° だった試行: 麻痺側 {(falls['q_paretic_at_fall'] < 0.5).sum()}/{len(falls)}, "
      f"非麻痺側 {(falls['q_nonparetic_at_fall'] < 0.5).sum()}/{len(falls)}")

## §D. policy間 vs policy内のばらつき(R1-2)

元の実験は、各条件・各policyにつき**1回の決定論的な試行**(環境seed固定、推論は平均行動)でした。
ここでは、1回のロールアウトの中で `num_envs=10` の並列環境を使い、**初期条件(開始位置 x,y ±0.5 m、落下高さ 0.01–0.05 m)を
10通り**に変えて反復します(向きは元と同じく固定)。任意で観測ノイズ(学習時と同じcorruption)も加えられます。

8seed × 2条件 × 5水準 × 10初期条件 = 800試行(ロールアウト80回)。途中から再開できます。

In [ ]:
VAR_LEVELS = [0.75, 0.70, 0.65, 0.60, 0.55]
N_IC = 10
OBS_NOISE = False          # True にすると観測ノイズも加える(別ファイルに保存)
VAR_CSV = OUT_DIR / ("D_variability_obsnoise.csv" if OBS_NOISE else "D_variability.csv")

done = set()
if VAR_CSV.exists():
    _d = pd.read_csv(VAR_CSV); done = set(zip(_d["condition"], _d["seed"], _d["level"]))

for cond, idx in HEMI_CONDITIONS.items():
    for seed in SEEDS:
        ckpt = latest_ckpt(seed)
        for level in VAR_LEVELS:
            if (cond, seed, level) in done:
                continue
            t0 = time.time()
            fs, _ = rollout_logged(ckpt, idx, level, num_envs=N_IC, obs_noise=OBS_NOISE, keep_traj=False)
            rows = [{"condition": cond, "seed": seed, "level": level, "ic": i,
                     "fell_over": bool(fs[i] > 0), "fall_step": int(fs[i]) if fs[i] > 0 else np.nan}
                    for i in range(N_IC)]
            pd.DataFrame(rows).to_csv(VAR_CSV, mode="a", header=not VAR_CSV.exists(), index=False)
            print(f"{cond} seed{seed} L={level}: falls {int((fs > 0).sum())}/{N_IC} ({time.time()-t0:.0f}s)")
print("完了")

## §D2. (任意)反復試行を他の速度にも拡張(R1-2, R2-1)

§Dと同じ設計(10初期条件)を、0.3・0.7・0.9 m/sでも行います。8seed × 2条件 × 5水準 × 3速度 = 240ロールアウト(2,400試行)。
これにより、「速度をまたいで左右差が再現するか」を、初期条件のばらつきも含めて検定できます(§E2)。
時間がかかるため、`RUN_D2 = True` にしたときだけ実行します。途中から再開できます。

In [ ]:
RUN_D2 = False
D2_SPEEDS = [0.3, 0.7, 0.9]
D2_CSV = OUT_DIR / "D2_variability_speeds.csv"

if RUN_D2:
    done = set()
    if D2_CSV.exists():
        _d = pd.read_csv(D2_CSV); done = set(zip(_d["condition"], _d["seed"], _d["level"], _d["speed_mps"]))
    for speed in D2_SPEEDS:
        for cond, idx in HEMI_CONDITIONS.items():
            for seed in SEEDS:
                ckpt = latest_ckpt(seed)
                for level in VAR_LEVELS:
                    if (cond, seed, level, speed) in done:
                        continue
                    t0 = time.time()
                    fs, _ = rollout_logged(ckpt, idx, level, speed=speed, num_envs=N_IC, keep_traj=False)
                    rows = [{"condition": cond, "seed": seed, "level": level, "speed_mps": speed, "ic": i,
                             "fell_over": bool(fs[i] > 0), "fall_step": int(fs[i]) if fs[i] > 0 else np.nan}
                            for i in range(N_IC)]
                    pd.DataFrame(rows).to_csv(D2_CSV, mode="a", header=not D2_CSV.exists(), index=False)
                    print(f"{speed} m/s {cond} seed{seed} L={level}: falls {int((fs > 0).sum())}/{N_IC} ({time.time()-t0:.0f}s)")
    print("完了")
else:
    print("RUN_D2 = False のためスキップしました")

## §E. 統計(R1-2)

In [ ]:
from scipy.stats import binomtest, wilcoxon

def per_seed_threshold_min_survived(df):
    # 元論文の定義: 転倒しなかった最も低い残存率
    out = {}
    for cond in HEMI_CONDITIONS:
        out[cond] = {}
        for s in SEEDS:
            sub = df[(df["condition"] == cond) & (df["seed"] == s)]
            surv = sub[~sub["fell_over"].astype(bool)]
            out[cond][s] = surv["level"].min() if len(surv) else np.inf
    return pd.DataFrame(out)

# (1) 元のfine sweep(あれば)を、元の定義で再集計し、符号検定
src = pd.read_csv("g1_hemiparesis_fine.csv") if Path("g1_hemiparesis_fine.csv").exists() else \
      mech[mech["level"] < 1.0]
thr = per_seed_threshold_min_survived(src)
print("=== seedごとの閾値(転倒しなかった最も低い残存率) ===")
print(thr.to_string()); print("平均:", thr.mean().round(4).to_dict())

# 非単調なseed(ある水準で転倒しないのに、それより高い水準で転倒する)の確認
for cond in HEMI_CONDITIONS:
    for s in SEEDS:
        sub = src[(src["condition"] == cond) & (src["seed"] == s)].sort_values("level")
        f = sub["fell_over"].astype(bool).values
        if any((~f[i]) and f[j] for i in range(len(f)) for j in range(i + 1, len(f))):
            print(f"  非単調: {cond} seed{s}: " + ", ".join(f"{l}:{'F' if x else '-'}" for l, x in zip(sub['level'], f)))

d = thr["right_half_paresis"] - thr["left_half_paresis"]
n_pos, n_neg, n_tie = int((d > 0).sum()), int((d < 0).sum()), int((d == 0).sum())
p_sign = binomtest(n_pos, n_pos + n_neg, 0.5).pvalue if n_pos + n_neg > 0 else np.nan
print(f"\n右>左 {n_pos}, 左>右 {n_neg}, 同着 {n_tie} → 正確符号検定(両側, 同着除外) p = {p_sign:.4f}")
nz = d[d != 0]
if len(nz):
    print(f"Wilcoxon符号順位検定(正確, 同着除外): p = {wilcoxon(nz, method='exact').pvalue:.4f}")

In [ ]:
# (2) §Dの反復試行から、policyごとの転倒確率・閾値・CI・分散分解
if VAR_CSV.exists():
    var = pd.read_csv(VAR_CSV)
    p = var.groupby(["condition", "seed", "level"])["fell_over"].mean().rename("p_fall").reset_index()
    print("=== policyごとの転倒確率(10初期条件) ===")
    print(p.pivot_table(index=["condition", "seed"], columns="level", values="p_fall").round(2).to_string())

    def thr50(sub):
        # 残存率を上げると転倒確率が下がる。p=0.5 を横切る残存率を線形補間で求める
        sub = sub.sort_values("level"); L = sub["level"].values; P = sub["p_fall"].values
        if (P < 0.5).all(): return L.min()        # 下限以下(打ち切り)
        if (P >= 0.5).all(): return L.max()       # 上限以上(打ち切り)
        for i in range(len(L) - 1):
            if P[i] >= 0.5 > P[i + 1]:
                return L[i] + (P[i] - 0.5) / (P[i] - P[i + 1]) * (L[i + 1] - L[i])
        return np.nan
    T50 = p.groupby(["condition", "seed"]).apply(thr50).unstack(0)
    print("\n=== policyごとの50%転倒閾値(補間) ===")
    print(T50.round(4).to_string())

    rng = np.random.default_rng(0)
    for cond in HEMI_CONDITIONS:
        x = T50[cond].dropna().values
        bs = [rng.choice(x, len(x), replace=True).mean() for _ in range(10000)]
        print(f"{cond}: 平均 {x.mean():.4f}, 95% bootstrap CI [{np.percentile(bs, 2.5):.4f}, {np.percentile(bs, 97.5):.4f}]")
    dd = (T50["right_half_paresis"] - T50["left_half_paresis"]).dropna()
    print(f"右−左 の差: 平均 {dd.mean():.4f}, 右>左 {int((dd>0).sum())}/{len(dd)}")
    nz = dd[dd.abs() > 1e-9]
    if len(nz):
        print(f"Wilcoxon符号順位検定(正確, 両側): p = {wilcoxon(nz, method='exact').pvalue:.4f}")

    # 分散分解: 各(条件, 水準)で、転倒(0/1)の全分散 = policy間分散 + policy内(初期条件)分散
    print("\n=== 分散分解: 転倒のばらつきのうち policy間 が占める割合 ===")
    rows = []
    for (cond, level), g in p.groupby(["condition", "level"]):
        between = g["p_fall"].var(ddof=0)
        within = (g["p_fall"] * (1 - g["p_fall"])).mean()
        tot = between + within
        rows.append({"condition": cond, "level": level, "mean_p": g["p_fall"].mean(),
                     "between": between, "within": within,
                     "frac_between": between / tot if tot > 0 else np.nan})
    print(pd.DataFrame(rows).round(3).to_string(index=False))
else:
    print("§Dを先に実行してください")

## §E2. 速度ごとの左右差の統計(R1-2)

(1) 元の速度依存CSV(`g1_hemiparesis_speed_dependence.csv`、各条件1回の試行)から、速度ごとに
policyごとの閾値(転倒しなかった最も低い残存率)を求め、左右差を符号検定します。
(2) §D2を実行していれば、反復試行から求めた50%閾値でも同じ検定を行います(0.5 m/sは§Dの結果を使います)。

In [ ]:
def thr50(sub):
    # 転倒確率が0.5を横切る残存率(線形補間)。全水準で<0.5なら下限、全水準で>=0.5なら上限(打ち切り)
    sub = sub.sort_values("level"); L = sub["level"].values; P = sub["p_fall"].values
    if (P < 0.5).all(): return L.min()
    if (P >= 0.5).all(): return L.max()
    for i in range(len(L) - 1):
        if P[i] >= 0.5 > P[i + 1]:
            return L[i] + (P[i] - 0.5) / (P[i] - P[i + 1]) * (L[i + 1] - L[i])
    return np.nan

def side_test(d, label):
    n_pos, n_neg, n_tie = int((d > 1e-9).sum()), int((d < -1e-9).sum()), int((d.abs() <= 1e-9).sum())
    p = binomtest(n_pos, n_pos + n_neg, 0.5).pvalue if n_pos + n_neg > 0 else np.nan
    print(f"{label}: 右>左 {n_pos}, 左>右 {n_neg}, 同着 {n_tie}, 正確符号検定 p = {p:.4f}, 差の平均 {d.mean():+.4f}")
    return {"label": label, "right_gt_left": n_pos, "left_gt_right": n_neg, "ties": n_tie, "p_sign": p, "mean_diff": d.mean()}

e2_rows = []
SP_CSV = Path("g1_hemiparesis_speed_dependence.csv")
if SP_CSV.exists():
    sp = pd.read_csv(SP_CSV)
    print("=== (1) 元の速度依存データ(各条件1回の試行)===")
    for speed in sorted(sp["speed_mps"].unique()):
        thr_s = per_seed_threshold_min_survived(sp[sp["speed_mps"] == speed])
        d = thr_s["right_half_paresis"] - thr_s["left_half_paresis"]
        e2_rows.append({"source": "original_single_rollout", "speed_mps": speed, **side_test(d, f"{speed} m/s")})
else:
    print("g1_hemiparesis_speed_dependence.csv が無いため(1)はスキップ")

frames = []
if VAR_CSV.exists():
    _v = pd.read_csv(VAR_CSV); _v["speed_mps"] = 0.5; frames.append(_v)
if D2_CSV.exists():
    frames.append(pd.read_csv(D2_CSV))
if frames:
    allv = pd.concat(frames, ignore_index=True)
    pv = allv.groupby(["speed_mps", "condition", "seed", "level"])["fell_over"].mean().rename("p_fall").reset_index()
    print("\n=== (2) 反復試行から求めた50%閾値 ===")
    for speed, g in pv.groupby("speed_mps"):
        T = g.groupby(["condition", "seed"]).apply(thr50).unstack(0)
        d = (T["right_half_paresis"] - T["left_half_paresis"]).dropna()
        row = side_test(d, f"{speed} m/s")
        nz = d[d.abs() > 1e-9]
        row["p_wilcoxon"] = wilcoxon(nz, method="exact").pvalue if len(nz) >= 1 else np.nan
        print(f"   Wilcoxon(正確): p = {row['p_wilcoxon']:.4f}")
        e2_rows.append({"source": "repeated_trials", "speed_mps": speed, **row})
if e2_rows:
    pd.DataFrame(e2_rows).to_csv(OUT_DIR / "E2_side_by_speed.csv", index=False)

## §F. 改訂原稿用の図(R1-2, R1-4)

- **図A**:用量反応曲線。§Dの反復試行から、policyごとの転倒確率(細線)と、8 policyの平均と95%ブートストラップ信頼区間(太線と帯)を、左右それぞれ描きます。
- **図B**:転倒前の関節角の時間経過。§Cの軌跡から、転倒時点で揃えて、デフォルト姿勢からのずれ(減衰した側/していない側、実際の関節角/指令値)を平均します。
  横の点線は、減衰なし(100%)で歩いているときの平均です。

色:右=青、左=オレンジ。白黒印刷でも区別できるよう、線種(実線/破線)も変えています。TIFF(300 dpi)とPNGで保存します。

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e5e5e5", "grid.linewidth": 0.6})
COL = {"right_half_paresis": "#2a78d6", "left_half_paresis": "#eb6834"}
LS = {"right_half_paresis": "-", "left_half_paresis": "--"}
LAB = {"right_half_paresis": "Right-side attenuation", "left_half_paresis": "Left-side attenuation"}
FIG_DIR = OUT_DIR / "figures"; FIG_DIR.mkdir(exist_ok=True)

def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=300, bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{name}.tiff", dpi=300, bbox_inches="tight")
    print("保存:", FIG_DIR / name)

# --- 図A:用量反応曲線 ---
if VAR_CSV.exists():
    var = pd.read_csv(VAR_CSV)
    p = var.groupby(["condition", "seed", "level"])["fell_over"].mean().rename("p_fall").reset_index()
    rng = np.random.default_rng(0)
    fig, ax = plt.subplots(figsize=(3.5, 2.8))
    for cond in HEMI_CONDITIONS:
        sub = p[p["condition"] == cond]
        for s, g in sub.groupby("seed"):
            g = g.sort_values("level")
            ax.plot(g["level"] * 100, g["p_fall"] * 100, color=COL[cond], ls=LS[cond], lw=0.6, alpha=0.35)
        piv = sub.pivot(index="seed", columns="level", values="p_fall")
        mean = piv.mean(0)
        bs = np.array([piv.iloc[rng.integers(0, len(piv), len(piv))].mean(0).values for _ in range(5000)])
        lo, hi = np.percentile(bs, 2.5, 0), np.percentile(bs, 97.5, 0)
        x = mean.index.values * 100
        ax.fill_between(x, lo * 100, hi * 100, color=COL[cond], alpha=0.15, lw=0)
        ax.plot(x, mean.values * 100, color=COL[cond], ls=LS[cond], lw=2, marker="o", ms=4, label=LAB[cond])
    ax.axhline(50, color="#999999", lw=0.8, ls=":")
    ax.set_xlabel("Remaining motor output α (%)")
    ax.set_ylabel("Fall probability (%)")
    ax.set_ylim(-3, 103)
    ax.legend(frameon=False, fontsize=8, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=1)
    save(fig, "figA_dose_response_ci"); plt.show()
else:
    print("§Dの結果が無いため図Aはスキップ")

# --- 図B:転倒前の関節角の時間経過 ---
if MECH_CSV.exists():
    mech = pd.read_csv(MECH_CSV)
    WIN = 50   # 転倒前1秒(50制御ステップ)
    fig, axes = plt.subplots(1, 2, figsize=(7, 2.8), sharey=True)
    for ax, cond in zip(axes, HEMI_CONDITIONS):
        idx = HEMI_CONDITIONS[cond]; msk = np.zeros(29, bool); msk[idx] = True
        curves = {"q_att": [], "q_non": [], "t_att": []}
        for _, r in mech[(mech["condition"] == cond) & (mech["fell_over"]) & (mech["level"] < 1.0)].iterrows():
            f = TRAJ_DIR / f"{cond}_seed{int(r['seed'])}_L{int(round(r['level']*100))}.npz"
            if not f.exists():
                continue
            tr = np.load(f); k = int(r["fall_step"])
            if k < WIN:
                continue
            dq = np.degrees(np.abs(tr["q"][0, k - WIN:k] - tr["q0"][0]))
            dt_ = np.degrees(np.abs(tr["target"][0, k - WIN:k] - tr["q0"][0]))
            curves["q_att"].append(dq[:, msk].mean(1)); curves["q_non"].append(dq[:, ~msk].mean(1))
            curves["t_att"].append(dt_[:, msk].mean(1))
        t = (np.arange(-WIN + 1, 1)) * 0.02
        n = len(curves["q_att"])
        if n:
            for key, lab, ls in [("q_att", "Attenuated side, actual", "-"),
                                 ("t_att", "Attenuated side, commanded", ":"),
                                 ("q_non", "Unattenuated joints, actual", "--")]:
                arr = np.array(curves[key]); mu = arr.mean(0); sd = arr.std(0)
                c = COL[cond] if key != "q_non" else "#777777"
                ax.plot(t, mu, color=c, ls=ls, lw=1.8, label=lab)
                ax.fill_between(t, mu - sd, mu + sd, color=c, alpha=0.12, lw=0)
        base = mech[(mech["condition"] == cond) & (mech["level"] == 1.0)]
        if len(base):
            ax.axhline(base["q_paretic_walk"].mean(), color=COL[cond], lw=0.8, ls=(0, (1, 2)))
            ax.axhline(base["q_nonparetic_walk"].mean(), color="#777777", lw=0.8, ls=(0, (1, 2)))
        ax.set_title(f"{LAB[cond]} (n = {n} falls)", fontsize=9)
        ax.set_xlabel("Time relative to fall onset (s)")
    axes[0].set_ylabel("|q − q⁰| (deg), mean ± SD")
    from matplotlib.lines import Line2D
    handles = [Line2D([], [], color="#333333", ls="-", lw=1.8, label="Attenuated side, actual"),
               Line2D([], [], color="#333333", ls=":", lw=1.8, label="Attenuated side, commanded"),
               Line2D([], [], color="#777777", ls="--", lw=1.8, label="Unattenuated joints, actual"),
               Line2D([], [], color="#777777", ls=(0, (1, 2)), lw=0.8, label="Mean during unattenuated walking (α = 100%)")]
    fig.legend(handles=handles, frameon=False, fontsize=7, ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.1))
    save(fig, "figB_prefall_joint_deviation"); plt.show()
else:
    print("§Cの結果が無いため図Bはスキップ")

## §G. GitHub公開用:各seedの最終checkpoint(R1-5)

解析で使っている「各seedの、最新の実行フォルダ」の中の、最終checkpointを確認し、公開用フォルダにコピーします。
あわせて、各seedの実行フォルダの一覧(古い実行フォルダが残っていないか)と、ファイルサイズを表示します。

In [ ]:
import shutil
REL_DIR = Path("release_checkpoints"); REL_DIR.mkdir(exist_ok=True)
rows = []
for seed in SEEDS:
    run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
    ck = latest_ckpt(seed)
    it = int(re.search(r"model_(\d+)", ck.name).group(1))
    dst = REL_DIR / f"g1_velocity_seed{seed}_{ck.name}"
    shutil.copy(ck, dst)
    rows.append({"seed": seed, "n_run_dirs": len(run_dirs), "used_run_dir": Path(run_dirs[-1]).name,
                 "final_checkpoint": ck.name, "iteration": it, "size_MB": round(ck.stat().st_size / 1e6, 2)})
ck_df = pd.DataFrame(rows)
print(ck_df.to_string(index=False))
print(f"\n合計サイズ: {ck_df['size_MB'].sum():.1f} MB → {REL_DIR}/ にコピーしました")
if (ck_df["iteration"] < 2999).any():
    print("⚠️ 3000 iterationsに届いていないseedがあります。使用している実行フォルダを確認してください。")
ck_df.to_csv(OUT_DIR / "G_checkpoints.csv", index=False)

## §H. 粗いスイープと速度依存スイープの測り直し(Table 1・Table 3の更新用)

元の結果CSVが見つからず、Table 2を改訂時の計算環境で測り直した値に更新したため、Table 1(粗いスイープ)と
Table 3(速度依存)も**同じ計算環境で測り直し**、すべての表を1つの環境の値でそろえます。

- 方法は元の実験と同じです:各条件・各policyにつき**1回の試行**(環境seed 100、初期の向き固定、250制御ステップ)。
  転倒の判定は§Cと同じ方法(終了条件を外し、体幹の傾き>70°をノートブック側で判定)です。
- 粗いスイープ:6水準(1.0, 0.8, 0.6, 0.4, 0.2, 0.0)× 8seed × 2条件 × 0.5 m/s = 96試行
- 速度依存:5水準(0.55〜0.75)× 4速度(0.3, 0.5, 0.7, 0.9 m/s)× 8seed × 2条件 = 320試行
- 途中から再開できます(T4で1時間前後)。0.5 m/sの結果は§Cと一致するはずで、その確認も行います。

In [ ]:
H_COARSE_CSV = OUT_DIR / "H_coarse.csv"
H_SPEED_CSV = OUT_DIR / "H_speed.csv"
H_COLS = ["condition", "seed", "level", "speed_mps", "fell_over", "fall_step"]
COARSE_LEVELS = [1.0, 0.8, 0.6, 0.4, 0.2, 0.0]
SPEED_LEVELS = [0.75, 0.70, 0.65, 0.60, 0.55]
SPEED_LIST = [0.3, 0.5, 0.7, 0.9]

def run_single_sweep(csv_path, levels, speeds):
    done = set()
    if csv_path.exists():
        _d = pd.read_csv(csv_path)
        done = set(zip(_d["condition"], _d["seed"], _d["level"], _d["speed_mps"]))
    total = len(HEMI_CONDITIONS) * len(SEEDS) * len(levels) * len(speeds)
    n = len(done)
    for speed in speeds:
        for cond, idx in HEMI_CONDITIONS.items():
            for seed in SEEDS:
                ckpt = latest_ckpt(seed)
                for level in levels:
                    if (cond, seed, level, speed) in done:
                        continue
                    t0 = time.time()
                    fs, _ = rollout_logged(ckpt, idx, level, speed=speed, keep_traj=False)
                    fell = bool(fs[0] > 0)
                    row = {"condition": cond, "seed": seed, "level": level, "speed_mps": speed,
                           "fell_over": fell, "fall_step": int(fs[0]) if fell else np.nan}
                    pd.DataFrame([row]).reindex(columns=H_COLS).to_csv(
                        csv_path, mode="a", header=not csv_path.exists(), index=False)
                    n += 1
                    print(f"[{n}/{total}] {speed} m/s {cond} seed{seed} L={level}: fell={fell} ({time.time()-t0:.0f}s)")

run_single_sweep(H_COARSE_CSV, COARSE_LEVELS, [0.5])
run_single_sweep(H_SPEED_CSV, SPEED_LEVELS, SPEED_LIST)
print("完了")

In [ ]:
hc = pd.read_csv(H_COARSE_CSV); hs = pd.read_csv(H_SPEED_CSV)
for d in (hc, hs):
    d["fell_over"] = d["fell_over"].astype(bool)

def fall_table(df):
    return (df.pivot_table(index="level", columns="condition", values="fell_over", aggfunc="mean")
              .sort_index(ascending=False)[["left_half_paresis", "right_half_paresis"]] * 100)

print("=== Table 1(更新):粗いスイープ、転倒率(%)、0.5 m/s、8 policy ===")
t1 = fall_table(hc); print(t1.round(1).to_string())

hs05 = hs[hs["speed_mps"] == 0.5]
print("\n=== Table 2(更新):精密スイープ、転倒率(%)、0.5 m/s、8 policy ===")
t2 = fall_table(hs05); print(t2.round(1).to_string())

# §Cの結果(0.5 m/s)と一致するかの確認(同じ条件なので、完全に一致するはず)
if MECH_CSV.exists():
    mc = pd.read_csv(MECH_CSV); mc = mc[mc["level"] < 1.0][["condition", "seed", "level", "fell_over", "fall_step"]]
    chk = hs05.merge(mc, on=["condition", "seed", "level"], suffixes=("_H", "_C"))
    same = (chk["fell_over_H"].astype(bool) == chk["fell_over_C"].astype(bool)).mean() * 100
    print(f"\n§Cとの一致(0.5 m/s、{len(chk)}試行):転倒の有無 {same:.1f}% 一致")

# Table 3:速度ごとの集団閾値(転倒率が50%以上になる最も高い残存率)と、policyごとの閾値の左右比較
rows = []
for speed in SPEED_LIST:
    sub = hs[hs["speed_mps"] == speed]
    fr = sub.groupby(["condition", "level"])["fell_over"].mean()
    thr_s = per_seed_threshold_min_survived(sub)
    d = thr_s["right_half_paresis"] - thr_s["left_half_paresis"]
    n_pos, n_neg, n_tie = int((d > 0).sum()), int((d < 0).sum()), int((d == 0).sum())
    p = binomtest(n_pos, n_pos + n_neg, 0.5).pvalue if n_pos + n_neg > 0 else np.nan
    row = {"speed_mps": speed}
    for cond in ["left_half_paresis", "right_half_paresis"]:
        over = fr[cond][fr[cond] >= 0.5]
        row[f"group_threshold_{cond}"] = over.index.max() * 100 if len(over) else "<55"
        row[f"mean_per_policy_threshold_{cond}"] = thr_s[cond].replace(np.inf, np.nan).mean() * 100
    row.update({"right_gt_left": n_pos, "left_gt_right": n_neg, "ties": n_tie, "p_sign": p})
    rows.append(row)
t3 = pd.DataFrame(rows)
print("\n=== Table 3(更新):速度ごとの集団閾値(%)と、policyごとの閾値の左右比較 ===")
print(t3.round(3).to_string(index=False))

t1.to_csv(OUT_DIR / "H_table1_coarse.csv"); t2.to_csv(OUT_DIR / "H_table2_fine.csv")
t3.to_csv(OUT_DIR / "H_table3_speed.csv", index=False)

# 図2(更新):速度ごとの用量反応曲線
import matplotlib.pyplot as plt
FIG_DIR = OUT_DIR / "figures"; FIG_DIR.mkdir(exist_ok=True)
SP_COL = {0.3: "#9ec5f4", 0.5: "#5598e7", 0.7: "#256abf", 0.9: "#0d366b"}   # 1色相の濃淡(速度=大きさ)
SP_MK = {0.3: "o", 0.5: "s", 0.7: "^", 0.9: "D"}   # 曲線が重なっても区別できるよう、速度ごとに印の形も変える
fig, axes = plt.subplots(1, 2, figsize=(7, 2.8), sharey=True)
for ax, cond, title in zip(axes, ["right_half_paresis", "left_half_paresis"],
                           ["Right-side attenuation", "Left-side attenuation"]):
    sub = hs[hs["condition"] == cond]
    for speed in SPEED_LIST:
        g = sub[sub["speed_mps"] == speed].groupby("level")["fell_over"].mean().sort_index()
        ax.plot(g.index * 100, g.values * 100, marker=SP_MK[speed], ms=5, mfc="none", mew=1.2, lw=1.5, color=SP_COL[speed], label=f"{speed} m/s")
    ax.axhline(50, color="#999999", lw=0.8, ls=":")
    ax.set_title(title, fontsize=9); ax.set_xlabel("Remaining motor output α (%)")
    ax.grid(True, color="#e5e5e5", lw=0.6); ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("Fall rate (%, 8 policies)")
axes[1].legend(frameon=False, fontsize=8, title="Speed", title_fontsize=8)
fig.savefig(FIG_DIR / "fig2_speed_dependence_remeasured.png", dpi=300, bbox_inches="tight")
fig.savefig(FIG_DIR / "fig2_speed_dependence_remeasured.tiff", dpi=300, bbox_inches="tight")
plt.show()

## 保存

In [ ]:
import shutil
shutil.make_archive("g1_hemiparesis_revision_outputs", "zip", OUT_DIR)
print("g1_hemiparesis_revision_outputs.zip を作成しました。このzipを共有してください。")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    shutil.copy("g1_hemiparesis_revision_outputs.zip", "/content/drive/MyDrive/")
    print("Google Driveにもコピーしました。")
except Exception as e:
    print("Driveへのコピーはスキップ:", e)